In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Browser Agent RL Orchestrator

**Audience:** You want a Colab-first walkthrough for browser-agent training with BrowserGym MiniWoB, optional OpenEnv probing, SFT, and GRPO.

**Prerequisites:** Basic Python, Colab, and high-level familiarity with LLM fine-tuning.

**Learning goals:**
- Inspect the `click-checkboxes` task interface.
- Run a baseline browser agent on the real MiniWoB task locally in Colab.
- Collect successful trajectories.
- Build an SFT dataset and prepare training.
- Prepare GRPO training from the SFT checkpoint.
- Save metrics and report artifacts cleanly.

**Outline:**
1. Validate Colab runtime and GPU.
2. Install BrowserGym MiniWoB, OpenEnv, and Unsloth dependencies.
3. Choose smoke/full profile and artifact root.
4. Bootstrap MiniWoB assets locally.
5. Run native probe and optional OpenEnv probe.
6. Run baseline evaluation and collect successful trajectories.
7. Build SFT dataset and optionally train SFT.
8. Optionally prepare and run GRPO.
9. Write final metrics/report artifacts.


## Step 1: Validate Runtime

This notebook assumes Colab with a single GPU. T4 is the primary target. The cell below fails fast if GPU is missing.

In [2]:
import os
import platform
import torch

print('Python:', platform.python_version())
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU required. Switch Colab runtime to GPU first.'
gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print('GPU:', gpu_name)
print(f'GPU memory: {gpu_mem_gb:.1f} GB')
assert any(name in gpu_name for name in ['T4', 'L4', 'A100']), f'Unexpected GPU for this notebook: {gpu_name}'

Python: 3.12.13
CUDA available: True
GPU: NVIDIA L4
GPU memory: 23.7 GB


## Step 2: Install Dependencies

These installs follow the current BrowserGym MiniWoB + OpenEnv + Unsloth Colab path. Re-run this cell if you restart the runtime.


In [3]:
from pathlib import Path
import os

# Set this manually if your repo is somewhere else in Colab.
PROJECT_ROOT = Path('/content/openEnvRL')

if not (PROJECT_ROOT / 'requirements.txt').exists():
    for candidate in [Path('/content/drive/MyDrive/openEnvRL'), Path('/content'), Path.cwd()]:
        if (candidate / 'requirements.txt').exists():
            PROJECT_ROOT = candidate
            break

assert (PROJECT_ROOT / 'requirements.txt').exists(), (
    f'requirements.txt not found. Set PROJECT_ROOT manually. Current guess: {PROJECT_ROOT}'
)
os.chdir(PROJECT_ROOT)
print('Using PROJECT_ROOT =', PROJECT_ROOT)

%pip install -q -r {str(PROJECT_ROOT / 'requirements.txt')}
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q "git+https://huggingface.co/spaces/openenv/browsergym_env"
%pip install -q playwright
!playwright install chromium


Using PROJECT_ROOT = /content/drive/MyDrive/openEnvRL
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.9/194.9 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.6/191.6 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.7/625.7 kB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 749.2/749.2 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 102.6 MB/

## Step 3: Drive-Backed Run Root

Every SFT rerun writes directly to one fresh Google Drive run directory. Code stays in `/content/openEnvRL`; never stage artifacts on ephemeral disk.

In [4]:
from datetime import datetime
from pathlib import Path

RUN_ID = datetime.now().strftime('%Y-%m-%d-%H%M%S-sft-smoke')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/openEnvRL/artifacts') / RUN_ID

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive artifact root:', ARTIFACT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Artifact root: /content/drive/MyDrive/browser-agent-rl-artifacts


## Step 4: Load Config and Choose Run Profile

Use `sft_smoke` first: exactly 2 successful trajectories and 2 SFT evaluation episodes. Use `full` only after smoke passes.

In [5]:
from src.config import ensure_runtime_dirs, load_config

config = load_config('configs/default.yaml')

RUN_PROFILE = 'sft_smoke'  # change to 'full' only after Drive smoke passes
if RUN_PROFILE == 'sft_smoke':
    config['default_eval_episodes'] = 2
    config['target_successes'] = 2
    config['max_steps_per_episode'] = 6
elif RUN_PROFILE == 'full':
    pass
else:
    raise ValueError(f'Unknown RUN_PROFILE: {RUN_PROFILE}')

config['env_backend'] = 'native_browsergym'
config['results_dir'] = str(ARTIFACT_ROOT / 'results')
config['data_dir'] = str(ARTIFACT_ROOT / 'data')
config['models_dir'] = str(ARTIFACT_ROOT / 'models')
config['artifact_root'] = str(ARTIFACT_ROOT)
config['miniwob_root'] = '/content/browsergym-assets'
config['sft_dataset_path'] = str(Path(config['data_dir']) / 'sft_dataset.jsonl')
config['sft_output_dir'] = str(Path(config['models_dir']) / 'sft_model')
config['sft_adapter_dir'] = str(Path(config['models_dir']) / 'sft_adapter')
config['grpo_output_dir'] = str(Path(config['models_dir']) / 'grpo_model')

ensure_runtime_dirs(config)
config


{'model_id': 'Qwen/Qwen3.5-0.8B',
 'task_id': 'click-checkboxes',
 'benchmark': 'miniwob',
 'env_backend': 'native_browsergym',
 'base_url': 'https://openenv-browsergym-env.hf.space',
 'miniwob_root': '/content/browsergym-assets',
 'results_dir': '/content/drive/MyDrive/browser-agent-rl-artifacts/results',
 'data_dir': '/content/drive/MyDrive/browser-agent-rl-artifacts/data',
 'models_dir': '/content/drive/MyDrive/browser-agent-rl-artifacts/models',
 'max_steps_per_episode': 6,
 'default_eval_episodes': 5,
 'final_eval_episodes': 500,
 'env_retry_attempts': 2,
 'target_successes': 5,
 'sft_dataset_path': '/content/drive/MyDrive/browser-agent-rl-artifacts/data/sft_dataset.jsonl',
 'sft_output_dir': '/content/drive/MyDrive/browser-agent-rl-artifacts/models/sft_model',
 'grpo_output_dir': '/content/drive/MyDrive/browser-agent-rl-artifacts/models/grpo_model'}

## Step 5: Bootstrap MiniWoB Assets

This clones the MiniWoB++ assets once and points BrowserGym at the real local `click-checkboxes` task.


In [6]:
from src.env.native_browsergym_task import bootstrap_miniwob_assets

miniwob_url = bootstrap_miniwob_assets(config['miniwob_root'])
config['miniwob_url'] = miniwob_url
print('MINIWOB_URL =', miniwob_url)


MINIWOB_URL = file:///content/browsergym-assets/miniwob-plusplus/miniwob/html/miniwob/


## Step 6: Native Probe

This is a lightweight schema sanity check. It helps confirm the task-shape assumptions before running the real local MiniWoB environment.


In [7]:
from src.pipeline.run_all import run_native_probe_phase

native_probe = run_native_probe_phase()
assert 'summary' in native_probe
assert 'instruction' in native_probe['summary']
native_probe

{'summary': {'instruction': 'Select Apple',
  'fields': {'target 0': 'Apple'},
  'dom_refs': [1],
  'raw_keys': ['dom_elements', 'fields', 'utterance']},
 'sample_action_attempt': {'action_type': 'CLICK', 'ref': 1}}

## Step 7: Optional OpenEnv Probe

This is only a diagnostic check for the remote OpenEnv BrowserGym client. Do not trust it for baseline metrics unless it binds to the requested task instead of the default click-test demo.


In [8]:
from src.pipeline.run_all import run_openenv_probe_phase

try:
    openenv_probe = run_openenv_probe_phase(config)
    print('OpenEnv instruction preview:', openenv_probe['instruction'][:200])
    print('OpenEnv observation preview:', openenv_probe['observation'][:500])
except Exception as exc:
    print('OpenEnv probe skipped/failed:', exc)


OpenEnv probe skipped/failed: OpenEnv did not bind to click-checkboxes. It appears to have fallen back to the default click-test task. Verify the remote BrowserGym space supports task_name on reset().


## Step 8: Manual Task Smoke Test

Run a tiny deterministic controller on the real local `click-checkboxes` task first. If this fails, the environment or action wiring is still wrong, so do not trust baseline model results yet.


In [ ]:
from src.pipeline.run_all import run_manual_smoke_phase

manual_smoke_metrics = run_manual_smoke_phase(config)
manual_smoke_metrics


## Step 9: Baseline Evaluation

This runs the pretrained policy on the local BrowserGym MiniWoB `click-checkboxes` task and writes baseline metrics JSON. Only trust this after the manual smoke test is green.


In [9]:
from src.pipeline.run_all import run_baseline_eval_phase

baseline_metrics = run_baseline_eval_phase(config)
baseline_metrics

Error: BrowserType.launch: Executable doesn't exist at /root/.cache/ms-playwright/chromium-1117/chrome-linux/chrome
╔════════════════════════════════════════════════════════════╗
║ Looks like Playwright was just installed or updated.       ║
║ Please run the following command to download new browsers: ║
║                                                            ║
║     playwright install                                     ║
║                                                            ║
║ <3 Playwright Team                                         ║
╚════════════════════════════════════════════════════════════╝

## Step 9: Drive-Backed SFT Phases

Run these script-first phases in order. Each phase reads/writes the same Drive run root; `sft_manifest` is final durability check.

In [ ]:
import subprocess
import sys

def run_drive_phase(phase):
    command = [
        sys.executable, 'scripts/run_phase.py', '--phase', phase,
        '--profile', RUN_PROFILE, '--artifact-root', str(ARTIFACT_ROOT),
    ]
    subprocess.run(command, check=True)

for phase in ['collect_successes', 'build_sft_dataset', 'sft_train', 'sft_eval', 'sft_manifest']:
    run_drive_phase(phase)

## Step 10: Inspect Required SFT Deliverables

Do not stop session until all six paths exist and manifest counts match JSONL records.

In [ ]:
from pathlib import Path

required_paths = [
    ARTIFACT_ROOT / 'data' / 'successful_trajectories.jsonl',
    ARTIFACT_ROOT / 'data' / 'sft_dataset.jsonl',
    ARTIFACT_ROOT / 'models' / 'sft_adapter',
    ARTIFACT_ROOT / 'models' / 'sft_model',
    ARTIFACT_ROOT / 'results' / 'sft_metrics.json',
    ARTIFACT_ROOT / 'run_manifest.json',
]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, f'Missing Drive artifacts: {missing}'
print('All required Drive artifacts saved.')

In [ ]:
import json

manifest = json.loads((ARTIFACT_ROOT / 'run_manifest.json').read_text())
print('collection:', manifest['collection'])
print('dataset:', manifest['dataset'])
print('SFT metrics:', manifest['metrics'])

## Step 10: Prepare and Optionally Run GRPO

Only enable this after SFT has produced a usable checkpoint and OpenEnv probing is stable.

In [ ]:
from src.train.grpo import build_grpo_job_config, run_grpo_training

grpo_config = build_grpo_job_config(config['sft_output_dir'], config['grpo_output_dir'])
grpo_config

In [ ]:
RUN_GRPO = False

if RUN_GRPO:
    grpo_stats = run_grpo_training(grpo_config)
    print(grpo_stats)
else:
    print('GRPO skipped. Set RUN_GRPO = True only after SFT checkpoint exists and OpenEnv runtime is verified.')

## Step 11: Final Report Skeleton

This writes a Markdown report to the artifact root. Replace placeholder SFT/GRPO metrics after those runs finish.

In [ ]:
from src.analysis.report import write_report

metrics = {
    'base': baseline_metrics,
    'sft': {'success_rate': 0.0, 'average_reward': 0.0, 'average_steps': 0.0},
    'grpo': {'success_rate': 0.0, 'average_reward': 0.0, 'average_steps': 0.0},
}
report_text = write_report(metrics, path=str(Path(config['results_dir']) / 'report.md'))
print(report_text[:1200])

## Exercise

Try this after the smoke run works:
- switch `RUN_PROFILE` from `smoke` to `full`
- increase `default_eval_episodes`
- compare baseline metrics before enabling SFT

Answer scaffold below records what changed and what broke, if anything.

In [ ]:
notes = {
    'profile': RUN_PROFILE,
    'what_changed': '',
    'baseline_observation': '',
    'openenv_issue': '',
}
notes

## Pitfall and Extension

**Common pitfall:** OpenEnv probe fails because the remote BrowserGym environment package/API drifted. If that happens, stop after Step 6 and patch the wrapper against the live package version instead of pushing ahead.

**Optional extension:** After the one-task pipeline works, duplicate the same flow for another MiniWoB task and compare how much baseline success rate changes before any retraining.